# Data extraction with Docling
En este notebook se extrae contenido de documentos pdf y se almacena como markdown

- **Markdown**: contenido de tipo texto 
- **Imágenes**: almacena páginas que contiene imágenes > 500x500
- **Tablas**: extraé con 2 párrafos de contexto + metadaata

### 1. Setup y configuracón

In [1]:
from pathlib import Path
from typing import List, Tuple

from docling_core.types.doc import PictureItem
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

d:\Cursos\Agentes\financial_deep_research_agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#path
OUTPUT_MD_DIR = "D:/Cursos/Agentes/financial_deep_research_agent/data/procesed/markdown"
OUTPUT_IMGES_DIR = "D:/Cursos/Agentes/financial_deep_research_agent/data/procesed/images"
OUTPUT_TABLES_DIR = "D:/Cursos/Agentes/financial_deep_research_agent/data/procesed/tables"
DATA_DIR = "D:/Cursos/Agentes/financial_deep_research_agent/data/raw"
LOG_DIR = "D:/Cursos/Agentes/financial_deep_research_agent/logs"

### 2. Exctract Metadata

In [51]:
def exctract_metada_from_filename(filename: str) -> dict:
    """Ectract metadata from filenama.
    Examples:
        - Amazon 10-Q Q1 2024.pdf
        - Microsoft 10-k 2023.pdf
    """
    filename = filename.replace(".pdf","").replace(".md","")
    parts = filename.split()

    return {
        "company_name": parts[0],
        "doc_type": parts[1],
        "fical_quarter": parts[2] if len(parts) == 4 else None,
        "fiscal_year": parts[-1] 
    }

## 3. Extract Markdown with docling

In [52]:
def convert_pdf_to_docling(pdf_file: Path, page_range):
    pipeline_options = PdfPipelineOptions()
    pipeline_options.images_scale = 2
    pipeline_options.generate_page_images = True
    pipeline_options.generate_picture_images = True

    doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    return doc_converter.convert(source=pdf_file,page_range=page_range)

In [53]:
#Save images
def save_page_images(doc_convert, img_dir: Path):
    """Find and save pages with large images (> 500x500 pix¿xels)"""
    pages_to_save = set()

    for item in doc_convert.document.iterate_items():
        element = item[0]
        if isinstance(element, PictureItem):
            image = element.get_image(doc_convert.document)

            if image.size[0] > 500 and image.size[1] > 500:
                page_no = element.prov[0].page_no if element.prov  else None

                if page_no:
                    pages_to_save.add(page_no)
        
        #save iamges
        for page_no in pages_to_save:
            page = doc_convert.document.pages[page_no]

            page.image.pil_image.save(img_dir/f"page_{page_no}.png", "PNG")

In [54]:
#save tables with context 
from docling_core.types.doc.document import TextItem, TableItem, SectionHeaderItem

def save_tables_with_context(doc_converter,table_dir:Path ,n_before = 2):
    """Extract tables with n context paragraphs/headers before"""
    elements = []
    for item in doc_converter.document.iterate_items():
        elements.append(item[0])
    table_count = 0
    for i, element in enumerate(elements):
        if isinstance(element, TableItem):
            table_count += 1
            page_no = element.prov[0].page_no if element.prov else None
            table_md = element.export_to_dataframe(doc=doc_converter.document)

            context = []
            count = 0
            for j in range(i-1,-1,-1):
                if count >= n_before:
                    break
                prev = elements[j]
                if isinstance(prev, (TextItem, SectionHeaderItem)):
                    context.insert(0, prev.text)
                    count += 1
            content_md = f"**Page:** {page_no}\n\n" + "\n\n".join(context) + "\n\n" + table_md.to_markdown(index=False)
            with open(table_dir / f"table_{table_count}_page_{page_no}.md", "w", encoding="utf-8") as f:
                f.write(content_md)

In [55]:
def append_text(md_dir: Path, pdf_file: Path, markdown_text: str):
    with open(md_dir / f"{pdf_file.stem}.md", "a", encoding="utf-8") as f:
        f.write(markdown_text)

In [56]:
from pypdf import PdfReader
import gc
reader = PdfReader(pdf_file)
pages_length = len(reader.pages)
pages_length

9

El documento es procesado cada 10 páginas, gc fuerza la limpieza de memoria, para almacenar va escribiendo en el documento con with, esto se hace con el fin de ahorro de memoria

In [57]:
def extract_pdf_content(pdf_file):
    for inicio in range(0, pages_length,10):
        fin = min(inicio + 10, pages_length)

        metadata = exctract_metada_from_filename(pdf_file.stem)

        company_name = metadata["company_name"]

        md_dir = Path(OUTPUT_MD_DIR) / company_name
        img_dir = Path(OUTPUT_IMGES_DIR) / company_name / pdf_file.stem
        table_dir = Path(OUTPUT_TABLES_DIR) / company_name / pdf_file.stem

        for dir_path in [md_dir, img_dir, table_dir]:
            dir_path.mkdir(parents = True, exist_ok=True)

        doc_converter = convert_pdf_to_docling(pdf_file, page_range=(inicio + 1, fin))
        markdown_text = doc_converter.document.export_to_markdown(page_break_placeholder="<!---page break--->")
        
        log_memory()

        # guaradar en markdown 
        append_text(md_dir, pdf_file ,markdown_text)

        #Save images
        save_page_images(doc_converter, img_dir)

        #Save tables
        save_tables_with_context(doc_converter, table_dir)

        del doc_converter
        gc.collect()
        log_memory()
        logging.info(f"Páginas {inicio+1}-{fin} escritas")

In [ ]:
# (md_dir / f"{pdf_file.stem}.md").write_text(markdown_text, encoding = "utf-8")

80271

In [58]:
import logging

def setup_logging(log_dir: Path, pdf_file: Path):
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{pdf_file.stem}.log"

    # Forzar reconfiguración
    logging.basicConfig(
        level=logging.INFO,
        format="[%(levelname)s] %(asctime)s %(name)s: %(message)s",
        handlers=[
            logging.FileHandler(log_path, encoding="utf-8"),
            logging.StreamHandler(),
        ],
        force=True,
    )
    return log_path

In [59]:
DATA_DIR

'D:/Cursos/Agentes/financial_deep_research_agent/data/raw'

In [60]:
import psutil
import os

def log_memory():
    process = psutil.Process(os.getpid())
    mem = process.memory_info().rss / 1024 / 1024  # MB
    logging.info(f"Memoria en uso: {mem:.1f} MB")

In [62]:
pdf_file = Path("D:/Cursos/Agentes/financial_deep_research_agent/data/raw/apple/apple 8-k q4 2023.pdf")

extract_pdf_content(pdf_file)
setup_logging(Path(LOG_DIR), pdf_file)

[INFO] 2026-05-24 09:20:11,512 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-24 09:20:11,517 [RapidOCR] download_file.py:60: File exists and is valid: D:\Cursos\Agentes\financial_deep_research_agent\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-24 09:20:11,519 [RapidOCR] main.py:57: Using D:\Cursos\Agentes\financial_deep_research_agent\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-24 09:20:13,432 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-24 09:20:13,433 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4/cls/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-24 09:20:15,468 [RapidOCR] download_file.py:82: Download size: 0.56MB
[INFO] 2026-05-24 09:20:15,904 [RapidOCR] download_file.py:95: Successfully saved to: D:\Cursos\Agentes\financial_deep_research_agent\.venv\Lib\site-packages\ra

WindowsPath('D:/Cursos/Agentes/financial_deep_research_agent/logs/apple 8-k q4 2023.log')

In [30]:
logging

<module 'logging' from 'C:\\Users\\DELL\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\logging\\__init__.py'>

In [61]:
data_path = Path(DATA_DIR)
pdf_files = data_path.rglob("*.pdf")

for idx, pdf_file in enumerate(pdf_files):
    # extract_pdf_content(pdf_file)
    print(pdf_file)

D:\Cursos\Agentes\financial_deep_research_agent\data\raw\amazon\amazon 10-q q1 2025.pdf
D:\Cursos\Agentes\financial_deep_research_agent\data\raw\amazon\amazon 10-q q2 2024.pdf
D:\Cursos\Agentes\financial_deep_research_agent\data\raw\amazon\amazon 10-q q2 2025.pdf
D:\Cursos\Agentes\financial_deep_research_agent\data\raw\apple\apple 10-k 2023.pdf
D:\Cursos\Agentes\financial_deep_research_agent\data\raw\apple\apple 10-k 2024.pdf
D:\Cursos\Agentes\financial_deep_research_agent\data\raw\apple\apple 10-q q1 2024.pdf
D:\Cursos\Agentes\financial_deep_research_agent\data\raw\apple\apple 10-q q2 2024.pdf
D:\Cursos\Agentes\financial_deep_research_agent\data\raw\apple\apple 10-q q4 2023.pdf
D:\Cursos\Agentes\financial_deep_research_agent\data\raw\apple\apple 8-k q4 2023.pdf
D:\Cursos\Agentes\financial_deep_research_agent\data\raw\google\google 10-k 2023.pdf
D:\Cursos\Agentes\financial_deep_research_agent\data\raw\google\google 10-k 2024.pdf
D:\Cursos\Agentes\financial_deep_research_agent\data\raw\